# **3. Model Training**

### <ins>Logistic Regression</ins>
&emsp;**a) Imports & Load Data**

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

# One shared definition of the categorical column list, imported by the
# feature engineering notebook and by ranking_system.py as well.
from features import CATEGORICAL_FEATURES

In [2]:
df = pd.read_csv('data/fr_en_features_complete_filled.csv',
                  keep_default_na=False, na_values=[''])
df.head()

,user_id,sent_id,token,POS,Dependency-Relation,Dependancy-Head,time,p_recall,sentence,nth_occurrence,...,lev_ratio,Definite,Gender,Number,Person,PronType,Mood,Tense,VerbForm,Reflex
0,YjS/mQOx,8XTyQUAl01,Le,DET,det,2,14.0,0,Le garçon,1,...,0.400000,Def,Masc,Sing,N/A,N/A,N/A,N/A,N/A,N/A
1,YjS/mQOx,8XTyQUAl01,garçon,NOUN,ROOT,0,14.0,0,Le garçon,1,...,0.222222,N/A,Masc,Sing,N/A,N/A,N/A,N/A,N/A,N/A
2,YjS/mQOx,8XTyQUAl02,Je,PRON,nsubj,4,14.0,0,Je suis une femme,1,...,0.000000,N/A,N/A,Sing,1,Prs,N/A,N/A,N/A,N/A
3,YjS/mQOx,8XTyQUAl02,suis,VERB,cop,4,14.0,0,Je suis une femme,1,...,0.285714,N/A,N/A,Sing,1,N/A,Ind,Pres,Fin,N/A
4,YjS/mQOx,8XTyQUAl02,une,DET,det,4,14.0,0,Je suis une femme,1,...,0.000000,Ind,Fem,Sing,N/A,Dem,N/A,N/A,N/A,N/A


&emsp;**b) Split data into train and test**
* I create temporal splits of the data to preserve the aspect of user history. For every user, I repeatedly peel off their *last-seen* sentence (in appearance order) into the test set, then their next-to-last, and so on, until the test set reaches 25% of all rows. Although, I never take more than half of any single user's sentences. This ensure that the model is trained on features that change over time and is test on predicting a users preformance as time/ practice continues.

In [3]:
# Create temporal train/test splits
# Each (user_id, sent_id) is one contiguous block of token rows (one practice instance).
# If the half-per-user cap is hit before reaching 25%, we stop there (25% is the max
# target/ test split percentage, yet it is permitted to not be a guarantee).
def temporal_train_test_split(df, test_size=0.25):
    df = df.reset_index(drop=True)
    total_rows = len(df)
    target_test = test_size * total_rows

    # Per sentence block: rank from the end (0 = last seen) and a per-user cap of half
    # the user's sentences (floor, so we never exceed half).
    blocks = df[['user_id', 'sent_id']].drop_duplicates()
    blocks['appear'] = blocks.groupby('user_id').cumcount()
    n_sents = blocks.groupby('user_id')['appear'].transform('size')
    blocks['from_end'] = n_sents - 1 - blocks['appear']
    blocks['cap'] = n_sents // 2
    blocks = blocks.set_index(['user_id', 'sent_id'])

    idx = df.set_index(['user_id', 'sent_id']).index
    from_end = pd.Series(idx.map(blocks['from_end']), index=df.index)
    cap = pd.Series(idx.map(blocks['cap']), index=df.index)

    # Peel one round (last sentence of every eligible user) at a time until we hit 25%
    # or every user has reached their half cap.
    test_mask = pd.Series(False, index=df.index)
    rank = 0
    max_cap = int(blocks['cap'].max())
    while test_mask.sum() < target_test and rank < max_cap:
        test_mask |= (from_end == rank) & (from_end < cap)
        rank += 1

    test_df = df[test_mask].reset_index(drop=True)
    train_df = df[~test_mask].reset_index(drop=True)
    return train_df, test_df


train_df, test_df = temporal_train_test_split(df)

total_rows = len(df)
target_test = 0.25 * total_rows
print(f'train: {len(train_df)} rows ({len(train_df)/total_rows:.2%})')
print(f'test:  {len(test_df)} rows ({len(test_df)/total_rows:.2%})')
print(f'reached 25% target: {len(test_df) >= target_test}')


train: 656753 rows (74.99%)
test:  218982 rows (25.01%)
reached 25% target: True


In [4]:
# Isolate Features for Training
train_df = train_df.drop(columns=['user_id','sent_id','token','sentence'])
test_df = test_df.drop(columns=['user_id','sent_id','token','sentence'])

&emsp;**c) Separate features (X) and labels (y)**
* It is also here that I break up the training and testing data further to run seperate models with access to different features. 1) A **Baseline** model without access to linguistic features and instead purely historical ones. 2) A **Complete** model with access to all features engineered prior, linguistic and historical. 3) A **Lingusitic Only** model with only access to linguistic features, for the sake of experimentation.

In [5]:
# Complete - all features 
X_train = train_df.drop(columns=['p_recall'])
y_train = train_df['p_recall']

X_test = test_df.drop(columns=['p_recall'])
y_test = test_df['p_recall']

In [6]:
# Baseline - w/o linguistic features
bl_train_df = train_df.drop(
    columns = ['POS','Dependency-Relation','Dependancy-Head','syllable_count','ortho_freq','lev_ratio','Definite','Gender','Number','Person','PronType','Mood','Tense','VerbForm','Reflex']
    )

bl_test_df = test_df.drop(
    columns= ['POS','Dependency-Relation','Dependancy-Head','syllable_count','ortho_freq','lev_ratio','Definite','Gender','Number','Person','PronType','Mood','Tense','VerbForm','Reflex']
)

X_bl_train = bl_train_df.drop(columns=['p_recall'])
y_bl_train = bl_train_df['p_recall']

X_bl_test = bl_test_df.drop(columns=['p_recall'])
y_bl_test = bl_test_df['p_recall']

In [7]:
# Isolated - linguistic only
ling_train_df = train_df.drop(
    columns = ['time','nth_occurrence','overall_recall_rate','token_recall_rate','token_streak']
)

ling_test_df = test_df.drop(
    columns = ['time','nth_occurrence','overall_recall_rate','token_recall_rate','token_streak']
)

X_ling_train = ling_train_df.drop(columns=['p_recall'])
y_ling_train = ling_train_df['p_recall']

X_ling_test = ling_test_df.drop(columns=['p_recall'])
y_ling_test = ling_test_df['p_recall']

&emsp;**d) Define any final preprocessing and train the models**


In [ ]:
# Define Preprocessing / Handling for Categorical Values
def get_preprocesser():
    return ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES),
        ],
        remainder="passthrough", # numeric features pass through unchanged
    )

In [9]:
# Initialize Model Piplines
complete_pipeline = Pipeline(steps=[
    ('preprocessing', get_preprocesser()),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

ling_pipeline = Pipeline(steps=[
    ('preprocessing', get_preprocesser()),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

bl_logRegr = LogisticRegression(max_iter=1000, class_weight='balanced') # no categorical values to preprocess

In [10]:
# Fit Baseline Model - includes no linguistic features
bl_logRegr.fit(X_bl_train, y_bl_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Defaul

In [ ]:
# Fit Pure Linguistic Model - includes only linguistic or token descriptive features 
ling_pipeline.fit(X_ling_train, y_ling_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](15,)","['POS','Dependency-Relation','Dependancy-Head',...,'Tense','VerbForm', 'Reflex']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,15
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all

In [12]:
# Fit Complete Model - includes all features
complete_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](20,)","['POS','Dependency-Relation','Dependancy-Head',...,'Tense','VerbForm', 'Reflex']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,20
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all

&emsp;**e) Evaluate the fit models**
* I am using **ROC-AUC** as a performance metric

&emsp;Scores without balanced weights:

* Baseline Model ROC-AUC: 0.7196
* Linguistics Only Model ROC-AUC: 0.6582
* Complete Model ROC-AUC: 0.7352


In [13]:
y_bl_pred_proba = bl_logRegr.predict_proba(X_bl_test)[:, 1]
bl_auc = roc_auc_score(y_bl_test, y_bl_pred_proba)
print(f"Baseline Model ROC-AUC: {bl_auc:.4f}")

y_ling_pred_proba = ling_pipeline.predict_proba(X_ling_test)[:, 1]
ling_auc = roc_auc_score(y_ling_test, y_ling_pred_proba)
print(f"Linguistics Only Model ROC-AUC: {ling_auc:.4f}")

y_complete_pred_proba = complete_pipeline.predict_proba(X_test)[:, 1]
complete_auc = roc_auc_score(y_test, y_complete_pred_proba)
print(f"Complete Model ROC-AUC: {complete_auc:.4f}")
 

Baseline Model ROC-AUC: 0.7199
Linguistics Only Model ROC-AUC: 0.6589
Complete Model ROC-AUC: 0.7370


### <ins>Gradient Boosted Tree</ins>
&emsp;**a) Define Pipelines and Train models**


In [18]:
from sklearn.preprocessing import OrdinalEncoder
from lightgbm import LGBMClassifier

In [ ]:
# Define unique preprocessor retrival function
cat_features = CATEGORICAL_FEATURES

def get_gbt_preprocesser():
    return ColumnTransformer(
        transformers=[
            ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_features),
        ],
        remainder="passthrough", # numeric features pass through unchanged
    )

# LightGBM indexes the frame the ColumnTransformer *emits*, not the one it
# receives. The transformer block is written out first, so the encoded
# categoricals occupy 0..10 and the passthrough numerics follow at 11+.
# I previously passed the pre-transform positions, which declared `time`,
# `overall_recall_rate`, `ortho_freq` and `lev_ratio` categorical while leaving
# Gender/Tense/Mood to be read as numbers -- that cost ~0.032 ROC-AUC.
CAT_IDX = list(range(len(cat_features)))  # -> [0, 1, ..., 10]

# NOTE: this has to be passed to .fit(), not to the constructor. LightGBM 4.6
# finds a constructor-level `categorical_feature` in params and discards it
# ("categorical_feature keyword has been found in `params` and will be
# ignored"), so the declaration silently does nothing when set that way.

# Initialize Model Piplines
complete_gbt_pipeline = Pipeline(steps=[
    ('preprocessing', get_gbt_preprocesser()),
    ('classifier', LGBMClassifier(class_weight='balanced', n_jobs=-1))
])

ling_gbt_pipeline = Pipeline(steps=[
    ('preprocessing', get_gbt_preprocesser()),
    ('classifier', LGBMClassifier(class_weight='balanced', n_jobs=-1))
])

bl_gbt = LGBMClassifier(class_weight='balanced', n_jobs=-1) # no categorical values to preprocess

In [21]:
# Fit Baseline Model - includes no linguistic features
bl_gbt.fit(X_bl_train, y_bl_train)

[LightGBM] [Info] Number of positive: 108110, number of negative: 548643
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002530 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 962
[LightGBM] [Info] Number of data points in the train set: 656753, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


,class_weight,'balanced'
,n_jobs,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,min_split_gain,0.0
,min_child_weight,0.001


In [ ]:
# Fit Pure Linguistic Model - includes only linguistic or token descriptive features
ling_gbt_pipeline.fit(X_ling_train, y_ling_train, classifier__categorical_feature=CAT_IDX)

In [ ]:
# Fit Complete Model - includes all features
complete_gbt_pipeline.fit(X_train, y_train, classifier__categorical_feature=CAT_IDX)

&emsp;**b) Evaluate the fit models**
* I am using **ROC-AUC** as a performance metric

In [ ]:
y_bl_gbt_pred_proba = bl_gbt.predict_proba(X_bl_test)[:, 1]
bl_gbt_auc = roc_auc_score(y_bl_test, y_bl_gbt_pred_proba)
print(f"Baseline Model ROC-AUC: {bl_gbt_auc:.4f}")

y_ling_gbt_pred_proba = ling_gbt_pipeline.predict_proba(X_ling_test)[:, 1]
ling_gbt_auc = roc_auc_score(y_ling_test, y_ling_gbt_pred_proba)
print(f"Linguistics Only Model ROC-AUC: {ling_gbt_auc:.4f}")

y_complete_gbt_pred_proba = complete_gbt_pipeline.predict_proba(X_test)[:, 1]
complete_gbt_auc = roc_auc_score(y_test, y_complete_gbt_pred_proba)
print(f"Complete Model ROC-AUC: {complete_gbt_auc:.4f}")

Results showcase that the strongest model is the light gradient boosted tree trained on both linguistic and temporal features. Yet, the margin over the best logistic regression model is slim, which is worth weighing against how much easier the linear model is to interpret.

### <ins>Serving Variant: No `time` Feature</ins>
&emsp;**a) Why a second model**
* `time` is how long the learner spent on the exercise, so it only exists **after** an attempt. A ranking system chooses the next card **before** the attempt, meaning `time` is simply unavailable at the moment the prediction is needed.
* Rather than feed the model a placeholder, I train a second complete model with `time` dropped. The AUC gap between the two is the cost of ranking honestly, and both get exported so the ranking system can be evaluated either way.

In [ ]:
# Complete feature set minus `time` (unknowable before the attempt)
X_train_nt = X_train.drop(columns=['time'])
X_test_nt  = X_test.drop(columns=['time'])

complete_nt_gbt_pipeline = Pipeline(steps=[
    ('preprocessing', get_gbt_preprocesser()),
    ('classifier', LGBMClassifier(class_weight='balanced', n_jobs=-1))
])

complete_nt_gbt_pipeline.fit(X_train_nt, y_train, classifier__categorical_feature=CAT_IDX)

In [ ]:
y_nt_pred_proba = complete_nt_gbt_pipeline.predict_proba(X_test_nt)[:, 1]
complete_nt_gbt_auc = roc_auc_score(y_test, y_nt_pred_proba)
print(f"Complete Model (no time) ROC-AUC: {complete_nt_gbt_auc:.4f}")
print(f"Complete Model (with time) ROC-AUC: {complete_gbt_auc:.4f}")
print(f"Cost of dropping `time`: {complete_gbt_auc - complete_nt_gbt_auc:.4f}")

### <ins>Export</ins>
&emsp;**a) Persist the fitted pipelines for `ranking_system.py`**
* Each pipeline is pickled with `joblib` **including its preprocessing step**, so serving is a single `predict_proba` call on a raw feature frame — no encoder needs rebuilding.
* Each model ships with a metadata file whose `feature_columns` field is the **column contract**. `remainder='passthrough'` binds the numeric features by *position*, so a frame with the right columns in the wrong order would predict quietly-wrong numbers instead of raising. `ranking_system.py` reindexes to this list before every call.
* A reminder recorded with the artifact: the label is 1 = **mistake**, so `predict_proba[:, 1]` is P(error) and a **high** score means a **hard**, high-priority card.

In [ ]:
import json
from pathlib import Path

import joblib
import lightgbm
import sklearn

Path('models').mkdir(exist_ok=True)

exports = [
    ('recall_gbt_complete', complete_gbt_pipeline,    X_train,    complete_gbt_auc),
    ('recall_gbt_no_time',  complete_nt_gbt_pipeline, X_train_nt, complete_nt_gbt_auc),
]

for name, pipe, X, auc in exports:
    joblib.dump(pipe, f'models/{name}.joblib')
    json.dump({
        'feature_columns': list(X.columns),         # THE column contract - order matters
        'categorical_features': cat_features,
        'target': 'p_recall',
        'target_meaning': '1 = learner made a mistake; predict_proba[:, 1] is P(error)',
        'higher_score_means': 'harder card / higher flashcard priority',
        'roc_auc': round(float(auc), 4),
        'n_train_rows': int(len(X)),
        'versions': {
            'lightgbm': lightgbm.__version__,
            'scikit-learn': sklearn.__version__,
        },
    }, open(f'models/{name}.metadata.json', 'w'), indent=2)
    print(f'{name}: {len(X.columns)} features, ROC-AUC {auc:.4f}')